In [1]:

!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy


name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-3862ba87-13e4-7a6a-b852-bc839875c437, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [2]:

GITHUB_USERNAME = "230103136-web"

assert GITHUB_USERNAME != "YOUR_GITHUB_USERNAME"
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")


Configured student GitHub identity: 230103136-web


In [3]:

%%writefile distributed_cuda_solver.py
import os
import sys
import json
import math
import hashlib
import subprocess
import re
import numpy as np
from mpi4py import MPI
from numba import cuda, float64

TILE_DIM = 16
SHARED_DIM = 18
GLOBAL_NX = GLOBAL_NY = 512
NY = NX = 256
STEPS = 400
LAMBDA_DAMP = 0.15
DX = DY = 1.0 / 513.0

DIRECTIONS = ("up", "down", "left", "right")
OPPOSITE = {
    "up": "down", "down": "up",
    "left": "right", "right": "left"
}
TAGS = {"up": 10, "down": 11, "left": 12, "right": 13}


# TASK 1 — MPI topology and hardware identity

def get_gpu_uuid():
    return subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid",
         "--format=csv,noheader"],
        text=True
    ).strip().splitlines()[0].strip()


def derive_student_seed(github_user, gpu_uuid):
    raw = f"{github_user.strip()}::{gpu_uuid}"
    digest = hashlib.sha256(raw.encode("utf-8")).hexdigest()
    return int(digest[:16], 16)


def init_distributed_context(github_user):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()

    if comm.Get_size() != 4:
        raise RuntimeError("Exactly 4 MPI ranks required")

    cuda.select_device(0)

    uuid = comm.bcast(
        get_gpu_uuid() if rank == 0 else None,
        root=0
    )

    seed = derive_student_seed(github_user, uuid)

    cart = comm.Create_cart(
        [2, 2],
        periods=[False, False],
        reorder=False
    )

    up, down = cart.Shift(0, 1)
    left, right = cart.Shift(1, 1)

    return {
        "comm": cart,
        "rank": cart.Get_rank(),
        "coords": cart.Get_coords(cart.Get_rank()),
        "neighbors": {
            "up": up, "down": down,
            "left": left, "right": right
        },
        "seed": seed,
        "uuid": uuid,
        "user": github_user
    }


# TASK 2 — Local subdomain initialization

def initialize_subdomains(ctx):
    rr, cc = ctx["coords"]

    y = (rr * NY + np.arange(NY + 2)) * DY
    x = (cc * NX + np.arange(NX + 2)) * DX
    xx, yy = np.meshgrid(x, y)

    phase = (ctx["seed"] % 10000) / 10000.0

    kappa = (
        0.16
        + 0.04 * np.sin(2 * np.pi * xx + phase)
        * np.cos(2 * np.pi * yy)
        + 0.04 * np.sin(4 * np.pi * yy + phase)
    )
    kappa = np.clip(kappa, 0.08, 0.24)

    cx = 0.25 + 0.10 * phase
    cy = 0.35 + 0.10 * (1 - phase)

    u = (
        (0.65 + 0.2 * phase)
        * np.exp(-((xx-cx)**2 + (yy-cy)**2)/0.012)
    )

    u += 0.35 * np.exp(
        -((xx-0.70)**2 + (yy-0.72)**2)/0.018
    )

    source = 0.002 * np.exp(
        -((xx-0.5)**2 + (yy-0.5)**2)/0.008
    )

    nbr = ctx["neighbors"]

    if nbr["up"] == MPI.PROC_NULL:
        u[0, :] = 0.0
    if nbr["down"] == MPI.PROC_NULL:
        u[-1, :] = 0.0
    if nbr["left"] == MPI.PROC_NULL:
        u[:, 0] = 0.0
    if nbr["right"] == MPI.PROC_NULL:
        u[:, -1] = 0.0

    return (
        cuda.to_device(u.astype(np.float64)),
        cuda.to_device(kappa.astype(np.float64)),
        cuda.to_device(source.astype(np.float64)),
        float(np.max(kappa[1:-1, 1:-1]))
    )


# TASK 3 — GPU halo packing and unpacking

@cuda.jit
def pack_halos_kernel(u, out, direction, n):
    q = cuda.grid(1)

    if q < n:
        if direction == 0:
            out[q] = u[1, q+1]
        elif direction == 1:
            out[q] = u[NY, q+1]
        elif direction == 2:
            out[q] = u[q+1, 1]
        else:
            out[q] = u[q+1, NX]


@cuda.jit
def unpack_halos_kernel(u, inp, direction, n):
    q = cuda.grid(1)

    if q < n:
        if direction == 0:
            u[0, q+1] = inp[q]
        elif direction == 1:
            u[NY+1, q+1] = inp[q]
        elif direction == 2:
            u[q+1, 0] = inp[q]
        else:
            u[q+1, NX+1] = inp[q]


def allocate_exchange():
    buffers = {}

    for d in DIRECTIONS:
        n = NX if d in ("up", "down") else NY

        buffers[d] = {
            "ds": cuda.device_array(n, dtype=np.float64),
            "dr": cuda.device_array(n, dtype=np.float64),
            "hs": cuda.pinned_array(n, dtype=np.float64),
            "hr": cuda.pinned_array(n, dtype=np.float64)
        }

    return buffers


def exchange_halos_async(ctx, u, buffers, stream):
    comm = ctx["comm"]
    nbr = ctx["neighbors"]

    for idx, d in enumerate(DIRECTIONS):
        if nbr[d] != MPI.PROC_NULL:
            n = NX if d in ("up", "down") else NY

            pack_halos_kernel[
                (n+127)//128, 128, stream
            ](u, buffers[d]["ds"], idx, n)

            buffers[d]["ds"].copy_to_host(
                buffers[d]["hs"], stream=stream
            )

    stream.synchronize()

    requests = []

    for d in DIRECTIONS:
        if nbr[d] != MPI.PROC_NULL:
            requests.append(
                comm.Irecv(
                    buffers[d]["hr"],
                    source=nbr[d],
                    tag=TAGS[OPPOSITE[d]]
                )
            )

    for d in DIRECTIONS:
        if nbr[d] != MPI.PROC_NULL:
            requests.append(
                comm.Isend(
                    buffers[d]["hs"],
                    dest=nbr[d],
                    tag=TAGS[d]
                )
            )

    return requests


def finish_exchange(ctx, u, buffers, stream, requests):
    MPI.Request.Waitall(requests)

    for idx, d in enumerate(DIRECTIONS):
        if ctx["neighbors"][d] != MPI.PROC_NULL:
            n = NX if d in ("up", "down") else NY

            buffers[d]["dr"].copy_to_device(
                buffers[d]["hr"], stream=stream
            )

            unpack_halos_kernel[
                (n+127)//128, 128, stream
            ](u, buffers[d]["dr"], idx, n)

    stream.synchronize()


# TASK 4 — Shared memory tiled CUDA stencil

@cuda.jit
def tiled_stencil_kernel(
    u, k, s, out, dt,
    rs, re, cs, ce, ring_only
):
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    r = rs + cuda.blockIdx.y * 16 + ty
    c = cs + cuda.blockIdx.x * 16 + tx

    sm = cuda.shared.array((18, 18), dtype=float64)

    valid = (
        r >= 1 and r <= NY
        and c >= 1 and c <= NX
    )

    if valid:
        sm[ty+1, tx+1] = u[r, c]
    else:
        sm[ty+1, tx+1] = 0.0

    if tx == 0:
        if valid:
            sm[ty+1, 0] = u[r, c-1]
        else:
            sm[ty+1, 0] = 0.0

    if tx == 15:
        if valid:
            sm[ty+1, 17] = u[r, c+1]
        else:
            sm[ty+1, 17] = 0.0

    if ty == 0:
        if valid:
            sm[0, tx+1] = u[r-1, c]
        else:
            sm[0, tx+1] = 0.0

    if ty == 15:
        if valid:
            sm[17, tx+1] = u[r+1, c]
        else:
            sm[17, tx+1] = 0.0

    cuda.syncthreads()

    if r <= re and c <= ce and valid:
        if (
            ring_only == 0
            or r == 1 or r == NY
            or c == 1 or c == NX
        ):
            z = sm[ty+1, tx+1]
            ku = k[r, c]

            flux_x = (
                0.5 * (ku+k[r, c+1])
                * (sm[ty+1, tx+2]-z)
                - 0.5 * (ku+k[r, c-1])
                * (z-sm[ty+1, tx])
            ) / (DX*DX)

            flux_y = (
                0.5 * (ku+k[r+1, c])
                * (sm[ty+2, tx+1]-z)
                - 0.5 * (ku+k[r-1, c])
                * (z-sm[ty, tx+1])
            ) / (DY*DY)

            out[r, c] = z + dt * (
                flux_x + flux_y
                - LAMBDA_DAMP*z*z*z
                + s[r, c]
            )


def launch_region(
    u, k, s, out, dt, stream,
    rs, re, cs, ce, ring
):
    gx = (ce-cs+16)//16
    gy = (re-rs+16)//16

    tiled_stencil_kernel[
        (gx, gy), (16, 16), stream
    ](
        u, k, s, out, dt,
        rs, re, cs, ce, int(ring)
    )


def step_overlapped(
    ctx, u, k, s, out, dt,
    buffers, sb, si, overlap=True
):
    requests = exchange_halos_async(
        ctx, u, buffers, sb
    )

    if overlap:
        launch_region(
            u, k, s, out, dt, si,
            2, NY-1, 2, NX-1, False
        )

        finish_exchange(
            ctx, u, buffers, sb, requests
        )

        launch_region(
            u, k, s, out, dt, sb,
            1, NY, 1, NX, True
        )

        si.synchronize()
        sb.synchronize()

    else:
        finish_exchange(
            ctx, u, buffers, sb, requests
        )

        launch_region(
            u, k, s, out, dt, sb,
            1, NY, 1, NX, False
        )

        sb.synchronize()

    return out, u


# TASK 5 — Reductions, benchmarking and PTX audit

def stats(ctx, u, previous):
    a = u.copy_to_host()[1:-1, 1:-1]

    local_energy = np.array(
        [np.sum(a)*DX*DY], dtype=np.float64
    )
    global_energy = np.zeros_like(local_energy)

    ctx["comm"].Allreduce(
        local_energy, global_energy, op=MPI.SUM
    )

    diff = (
        np.sum((a-previous)**2)*DX*DY
        if previous is not None else 0.0
    )

    local = np.array([diff], dtype=np.float64)
    total = np.zeros_like(local)

    ctx["comm"].Allreduce(
        local, total, op=MPI.SUM
    )

    return (
        float(global_energy[0]),
        float(np.sqrt(total[0])),
        a
    )


def run_mode(ctx, mode, dt):
    u, k, s, kmax = initialize_subdomains(ctx)

    out = cuda.to_device(
        np.zeros((NY+2, NX+2), dtype=np.float64)
    )

    buffers = allocate_exchange()
    sb = cuda.stream()
    si = cuda.stream()

    trace = []
    previous = None

    ctx["comm"].Barrier()
    start = MPI.Wtime()

    for step in range(STEPS+1):
        if step % 100 == 0:
            energy, residual, previous = stats(
                ctx, u, previous
            )

            trace.append({
                "step": step,
                "energy": energy,
                "residual": residual
            })

            if ctx["rank"] == 0:
                print(
                    f"{mode} step={step} "
                    f"energy={energy:.15e} "
                    f"residual={residual:.15e}",
                    flush=True
                )

        if step == STEPS:
            break

        u, out = step_overlapped(
            ctx, u, k, s, out, dt,
            buffers, sb, si,
            mode == "overlapped"
        )

    ctx["comm"].Barrier()
    elapsed = MPI.Wtime()-start

    wall = ctx["comm"].allreduce(
        elapsed, op=MPI.MAX
    )

    return u, trace, wall*1000, kmax


def audit_ptx(ctx, u, k, s, dt):
    out = cuda.device_array_like(u)

    launch_region(
        u, k, s, out, dt,
        cuda.default_stream(),
        1, NY, 1, NX, False
    )

    cuda.synchronize()

    sig = list(tiled_stencil_kernel.signatures)[-1]
    asm = tiled_stencil_kernel.inspect_asm(sig)

    counts = {}

    for name in (
        "st.shared.f64",
        "ld.shared.f64",
        "bar.sync"
    ):
        counts[name] = len(
            re.findall(
                r"\b"+re.escape(name)+r"\b", asm
            )
        )

    for prefix in ("st", "ld"):
        counts[prefix+".shared_total"] = len(
            re.findall(
                r"\b"+prefix+r"\.shared(?:\.[\w]+)*",
                asm
            )
        )

    counts["bar.sync_total"] = len(
        re.findall(r"\bbar\.sync\b", asm)
    )

    counts["passed"] = (
        counts["st.shared_total"] > 0
        and counts["ld.shared_total"] > 0
        and counts["bar.sync_total"] > 0
    )

    return counts


def run_solver(username):
    ctx = init_distributed_context(username)

    if ctx["rank"] == 0:
        print(
            "mpi_ranks=4 topology=2x2 "
            "GPU_UUID="+ctx["uuid"]+" "
            "INSTANCE_SEED="+str(ctx["seed"]),
            flush=True
        )

    u, k, s, kmax = initialize_subdomains(ctx)

    kmax = ctx["comm"].allreduce(
        kmax, op=MPI.MAX
    )

    dtmax = DX*DX/(4*kmax)
    dt = 0.8*dtmax

    if ctx["rank"] == 0:
        print(
            f"kappa_max={kmax:.15e} "
            f"dx={DX:.15e} "
            f"dt_max={dtmax:.15e} "
            f"dt={dt:.15e}",
            flush=True
        )

    ptx = audit_ptx(ctx, u, k, s, dt)

    if ctx["rank"] == 0:
        print(
            "PTX_AUDIT "+json.dumps(ptx),
            flush=True
        )

    us, ts, ms, _ = run_mode(
        ctx, "serialized", dt
    )

    uo, to, mo, _ = run_mode(
        ctx, "overlapped", dt
    )

    aa = us.copy_to_host()[1:-1, 1:-1]
    bb = uo.copy_to_host()[1:-1, 1:-1]

    local = float(np.max(np.abs(aa-bb)))

    parity = ctx["comm"].allreduce(
        local, op=MPI.MAX
    )

    if ctx["rank"] == 0:
        manifest = {
            "github_username": username,
            "gpu_uuid": ctx["uuid"],
            "instance_seed": ctx["seed"],
            "mpi_ranks": 4,
            "cartesian_dims": [2, 2],
            "grid": [512, 512],
            "local_padded_shape": [258, 258],
            "steps": STEPS,
            "kappa_max": kmax,
            "dx": DX,
            "dt": dt,
            "dt_max": dtmax,
            "serialized_ms": ms,
            "overlapped_ms": mo,
            "parity_max_diff": parity,
            "ptx_audit": ptx,
            "ptx_audit_passed": ptx["passed"],
            "serialized_energy_trace": ts,
            "overlapped_energy_trace": to
        }

        with open(
            "submission_manifest.json", "w"
        ) as f:
            json.dump(manifest, f, indent=2)

        print(
            "FINAL_MANIFEST "+json.dumps(manifest),
            flush=True
        )

        passed = (
            parity < 1e-12 and ptx["passed"]
        )

        print(
            "RESULT "+(
                "PASS" if passed
                else "CHECK_REQUIRED"
            ),
            flush=True
        )

    ctx["comm"].Barrier()


if __name__ == "__main__":
    if len(sys.argv) < 2:
        raise SystemExit(
            "Usage: python distributed_cuda_solver.py "
            "GITHUB_USERNAME"
        )

    run_solver(sys.argv[1])


Writing distributed_cuda_solver.py


In [4]:

import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME
]

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    timeout=1200
)

print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(
        f"MPI execution failed: {proc.returncode}"
    )


mpi_ranks=4 topology=2x2 GPU_UUID=GPU-3862ba87-13e4-7a6a-b852-bc839875c437 INSTANCE_SEED=7639941702540601062
kappa_max=2.316279347060295e-01 dx=1.949317738791423e-03 dt_max=4.101232059498211e-06 dt=3.280985647598569e-06
PTX_AUDIT {"st.shared.f64": 5, "ld.shared.f64": 5, "bar.sync": 1, "st.shared_total": 10, "ld.shared_total": 5, "bar.sync_total": 1, "passed": true}
serialized step=0 energy=4.503866966629481e-02 residual=0.000000000000000e+00
serialized step=100 energy=4.502048271677786e-02 residual=9.643740970029775e-04
serialized step=200 energy=4.500920711145008e-02 residual=9.386942830135598e-04
serialized step=300 energy=4.499902516168916e-02 residual=9.210748229533373e-04
serialized step=400 energy=4.498924906375767e-02 residual=9.043611202854570e-04
overlapped step=0 energy=4.503866966629481e-02 residual=0.000000000000000e+00
overlapped step=100 energy=4.502048271677786e-02 residual=9.643740970029775e-04
overlapped step=200 energy=4.500920711145008e-02 residual=9.386942830135598e

### Analytical & Forensic Engineering Responses

**Q1. Memory Coalescing & Strided Column Packing**

The local array has a shape of 258 × 258 and uses float64 values (8 bytes each). In row-major memory, consecutive elements within a row are contiguous, while consecutive elements in a column are separated by 258 × 8 = 2,064 bytes.

Therefore, transferring an entire column directly as one contiguous DMA operation is not possible without including unrelated elements.

Each halo column contains 256 values, requiring only 2,048 bytes. A rectangular bounding-box transfer covering 256 complete rows would require 256 × 258 × 8 = 528,384 bytes. This would introduce 526,336 unnecessary bytes per column, or 1,052,672 extra bytes for both columns.

Our CUDA packing kernels avoid this overhead by collecting the required column elements into contiguous device buffers before transferring them to pinned host memory.

**Q2. Shared-Memory Tile Indexing & Bank Conflict Analysis**

Our implementation uses 16 × 16 CUDA thread blocks and an 18 × 18 shared-memory tile, including a one-cell halo.

Thread (tx=0, ty=0) stores its center value at smem[1,1]. It also loads the left halo into smem[1,0] and the upper halo into smem[0,1].

Thread (tx=15, ty=15) stores its center value at smem[16,16]. It loads the right halo into smem[16,17] and the lower halo into smem[17,16].

On the Tesla T4, shared memory has 32 banks with a 4-byte bank width. Each float64 value occupies 8 bytes and spans two 4-byte banks.

Row-wise access by adjacent threads generally distributes consecutive double-precision values across bank pairs efficiently. Column-wise access has a stride of 18 × 8 = 144 bytes, producing repeated bank mappings and potentially multiple-way bank conflicts.

Bank conflicts depend on the actual warp access pattern, not simply on using float64. The 18-element row stride can cause conflicts during column-oriented accesses.

**Q3. PTX Assembly Audit & Warp Synchronization**

The actual PTX audit produced the following results:

- st.shared.f64: 5
- ld.shared.f64: 5
- bar.sync: 1
- Total shared-memory stores: 10
- Total shared-memory loads: 5
- PTX audit passed: true

The 16 × 16 thread block contains 256 threads, organized into 8 warps.

The cuda.syncthreads() instruction ensures that all threads finish loading their shared-memory elements and halo values before stencil calculations begin.

Without this synchronization, one warp could read a neighboring value before another warp has written it. This read-after-write hazard could produce incorrect or nondeterministic results.

The presence of bar.sync in the PTX confirms that block-wide synchronization was compiled into the kernel.

**Q4. Multi-Process Context Sharing on a Single Physical GPU**

Our experiment launched four MPI processes sharing one NVIDIA Tesla T4 GPU.

Each process creates its own CUDA context. The GPU driver schedules execution from these contexts across the available streaming multiprocessors and manages access to memory-transfer engines.

The measured execution times were:

- Serialized execution: 2077.536 ms
- Overlapped execution: 1464.758 ms
- Speedup: approximately 1.418×
- Execution-time reduction: approximately 29.5%

The overlapped implementation performs independent interior computations while MPI halo communication is in progress.

However, all four processes share the same physical GPU, which creates resource contention and limits concurrency. Therefore, the observed speedup cannot be interpreted as equivalent to four independent GPUs.

On a multi-node system with dedicated GPUs and InfiniBand communication, computation and communication could overlap differently because GPU resources would not be shared in the same way.

**Q5. Point-to-Point Message Deadlock & MPI.PROC_NULL Semantics**

Replacing non-blocking MPI.Isend and MPI.Irecv operations with blocking MPI.Send followed by MPI.Recv can cause deadlock.

For small messages, MPI may use an eager protocol that temporarily buffers outgoing data. For larger messages, MPI may switch to a rendezvous protocol, requiring the receiving process to participate before the send can complete.

If neighboring ranks all perform blocking sends before posting their corresponding receives, they may wait indefinitely for one another.

Our implementation avoids this problem by posting non-blocking receives and sends, followed by MPI.Request.Waitall().

MPI.PROC_NULL represents a nonexistent neighbor. Communication operations involving MPI.PROC_NULL complete without actual data transfer.

For example, Rank (0,0) has no upper or left neighbor in our non-periodic 2 × 2 topology. MPI.PROC_NULL allows these boundaries to be handled without communicating with nonexistent processes.

The physical boundaries maintain homogeneous Dirichlet conditions, u = 0.

**Q6. Explicit CFL Stability & Energy Conservation Proof**

Our simulation recorded the following parameters:

- Maximum diffusivity: 0.23162793470602952
- Grid spacing: Δx = 1/513 ≈ 0.00194931774
- Stability limit: Δtmax = 4.1012320595 × 10⁻⁶
- Selected time step: Δt = 3.2809856476 × 10⁻⁶

The explicit diffusion stability condition is:

Δt ≤ Δx² / (4κmax)

Our selected time step is 80% of the calculated stability limit:

Δt / Δtmax = 0.8

Therefore, the selected time step satisfies the stated diffusion CFL condition.

The global energy values obtained using MPI.Allreduce were:

| Step | Global Energy |
|---|---|
| 0 | 0.045038669666294814 |
| 100 | 0.04502048271677786 |
| 200 | 0.04500920711145008 |
| 300 | 0.04499902516168916 |
| 400 | 0.044989249063757675 |

The global energy decreases monotonically across all recorded checkpoints.

The total decrease is approximately 0.000049421, corresponding to about 0.11% of the initial energy.

This behavior is consistent with dissipative evolution. However, because the model contains a source term and nonlinear damping, strict energy conservation is not expected.

Finally, the serialized and overlapped implementations produced a maximum absolute difference of 0.0, satisfying the required tolerance of 10⁻¹².

**Conclusion**

The implementation successfully completed 400 simulation steps using four MPI ranks and a shared NVIDIA Tesla T4 GPU.

The PTX audit passed, the two execution modes produced identical numerical results, and the overlapped implementation reduced measured execution time by approximately 29.5%.

The recorded results support the correctness of the tested numerical implementation and demonstrate the performance benefit of overlapping computation with communication in this experiment.
